# Yoruba and Hausa ASR training

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
PROJECT_DIR = '/content/drive/MyDrive/speech-to-text-system'
%cd $PROJECT_DIR
!pip install -q -r requirements.txt
import sys
sys.path.insert(0, PROJECT_DIR)

In [ ]:
from pathlib import Path
import torch
RAW_DIR = Path(PROJECT_DIR) / 'data' / 'raw'
DATASET_DIR = Path(PROJECT_DIR) / 'data' / 'processed' / 'common_voice'
OUTPUT_DIR = Path('/content/drive/MyDrive/yoruba-hausa-asr/checkpoints')
MODEL_NAME = 'openai/whisper-base'
EPOCHS = 1.0
TRAIN_BATCH_SIZE = 2
EVAL_BATCH_SIZE = 2
GRADIENT_ACCUMULATION_STEPS = 8
GRADIENT_CHECKPOINTING = True
print('CUDA available:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'Enable a Colab GPU runtime before training.'

In [ ]:
from src.data_loader import build_dataset_dict
if not DATASET_DIR.exists():
    dataset = build_dataset_dict('mozilla-foundation/common_voice_17_0', raw_dir=RAW_DIR, revision=None, splits=['train', 'validation', 'test'], min_duration=0.5, max_duration=30.0, min_rms=1e-4, preserve_yoruba_diacritics=True, seed=42, balance_train=True)
    DATASET_DIR.parent.mkdir(parents=True, exist_ok=True)
    dataset.save_to_disk(str(DATASET_DIR))
else:
    print('Using existing processed dataset:', DATASET_DIR)

In [ ]:
from argparse import Namespace
from scripts.train import latest_checkpoint, train_model
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
training_args = Namespace(dataset_dir=DATASET_DIR, model_name=MODEL_NAME, output_dir=OUTPUT_DIR, epochs=EPOCHS, learning_rate=1e-4, train_batch_size=TRAIN_BATCH_SIZE, eval_batch_size=EVAL_BATCH_SIZE, gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS, gradient_checkpointing=GRADIENT_CHECKPOINTING, seed=42, resume_from_checkpoint=latest_checkpoint(OUTPUT_DIR), lora_rank=16, lora_alpha=32, lora_dropout=0.05)
trainer = train_model(training_args)

In [ ]:
REPORT_PATH = OUTPUT_DIR / 'test_metrics.json'
!python scripts/evaluate.py --dataset-dir $DATASET_DIR --model-dir $OUTPUT_DIR --base-model $MODEL_NAME --batch-size 2 --output $REPORT_PATH